In [ ]:
#  Mount Google Drive ===


# === Imports ===
# === Imports ===
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

# === Paths ===
dataset_path = "C:/Users/ganga/Downloads/dataset_blood_group/dataset_blood_group"  # ⚠️ Adjust if needed
train_path = "C:/Users/ganga/train_dataset"
test_path = "C:/Users/ganga/test_dataset"
model_save_dir = "C:/Users/ganga/saved_models"
model_save_path = os.path.join(model_save_dir, "new_model.keras")

os.makedirs(train_path, exist_ok=True)
os.makedirs(test_path, exist_ok=True)
os.makedirs(model_save_dir, exist_ok=True)

# === Helper: Check valid image ===
def is_image_file(filename):
    return filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))

# === Split Dataset ===
if not os.path.exists(os.path.join(train_path, "A")):
    print("Splitting dataset into train/test...")

    categories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    train_ratio = 0.8

    for category in categories:
        src_folder = os.path.join(dataset_path, category)
        train_folder = os.path.join(train_path, category)
        test_folder = os.path.join(test_path, category)

        os.makedirs(train_folder, exist_ok=True)
        os.makedirs(test_folder, exist_ok=True)

        images = [img for img in os.listdir(src_folder) if is_image_file(img)]
        random.shuffle(images)

        train_size = int(len(images) * train_ratio)
        train_images = images[:train_size]
        test_images = images[train_size:]

        for img in train_images:
            src = os.path.join(src_folder, img)
            dst = os.path.join(train_folder, img)
            shutil.copy(src, dst)

        for img in test_images:
            src = os.path.join(src_folder, img)
            dst = os.path.join(test_folder, img)
            shutil.copy(src, dst)

    print("✅ Dataset split complete.")
else:
    print("✅ Dataset already split.")

# === Preprocessing ===
def preprocess_image(image_path, target_size=(96, 103)):
    img = load_img(image_path, target_size=target_size, color_mode="grayscale")
    img_array = img_to_array(img) / 255.0
    img_array = np.repeat(img_array, 3, axis=-1)
    return img_array

def load_data_from_directory(directory):
    X, y = [], []
    class_labels = sorted(os.listdir(directory))
    label_map = {label: idx for idx, label in enumerate(class_labels)}

    print(f"Class mapping: {label_map}")

    for label in class_labels:
        label_path = os.path.join(directory, label)

        for img_file in os.listdir(label_path):
            if not is_image_file(img_file):
                continue

            img_path = os.path.join(label_path, img_file)

            try:
                X.append(preprocess_image(img_path))
                y.append(label_map[label])
            except Exception as e:
                print(f"Error loading {img_path}: {e}")

    return np.array(X), np.array(y), label_map

# === Load Data ===
X_train, y_train, train_label_map = load_data_from_directory(train_path)
X_test, y_test, test_label_map = load_data_from_directory(test_path)
num_classes = len(set(y_train))

# === Class Weights ===
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: w for i, w in enumerate(class_weights)}

# === Learning Rate Scheduler ===
def lr_schedule(epoch):
    initial_lr = 1e-3
    drop = 0.5
    epochs_drop = 5
    return initial_lr * (drop ** (epoch // epochs_drop))

# === Model ===
def build_model(num_classes):
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(96,103,3))
    base_model.trainable = True

    inputs = Input(shape=(96,103,3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# === Train ===
model = build_model(num_classes)

callbacks = [
    LearningRateScheduler(lr_schedule),
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ModelCheckpoint(model_save_path, monitor="val_accuracy", save_best_only=True, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    class_weight=class_weights_dict
)

# === Evaluate ===
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Test Accuracy: {acc * 100:.2f}%")

# === Plot ===
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title("Accuracy")
plt.show()

# === Confusion Matrix ===
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes))

conf_matrix = confusion_matrix(y_test, y_pred_classes)
ConfusionMatrixDisplay(conf_matrix, display_labels=test_label_map.keys()).plot()
plt.show()
# import os
# import shutil
# import random
# import numpy as np
# import matplotlib.pyplot as plt
# import tensorflow as tf
# from tensorflow.keras.applications import EfficientNetB0
# from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
# from tensorflow.keras.models import Model, load_model
# from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping, ModelCheckpoint
# from tensorflow.keras.preprocessing.image import load_img, img_to_array
# from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
# from sklearn.utils.class_weight import compute_class_weight

# # === Paths ===
# dataset_path = "C:/Users/ganga/Downloads/dataset_blood_group"
# train_path = "C:/Users/ganga/train_dataset"
# test_path = "C:/Users/ganga/test_dataset"
# model_save_path = "C:/Users/ganga/saved_models/new_model.keras"
# os.makedirs("C:/Users/ganga/saved_models", exist_ok=True)

# # === Split Dataset (Only run ONCE or delete folders before re-running) ===
# if not os.path.exists(train_path) or not os.path.exists(test_path):
#     print("Splitting dataset into train/test...")

#     categories = os.listdir(dataset_path)
#     train_ratio = 0.8

#     for category in categories:
#         os.makedirs(os.path.join(train_path, category), exist_ok=True)
#         os.makedirs(os.path.join(test_path, category), exist_ok=True)

#         images = os.listdir(os.path.join(dataset_path, category))
#         random.shuffle(images)
#         train_size = int(len(images) * train_ratio)
#         train_images = images[:train_size]
#         test_images = images[train_size:]

#         for img in train_images:
#             shutil.copy(os.path.join(dataset_path, category, img), os.path.join(train_path, category, img))
#         for img in test_images:
#             shutil.copy(os.path.join(dataset_path, category, img), os.path.join(test_path, category, img))

#     print("✅ Dataset split complete.")
# else:
#     print("✅ Dataset already split.")

# # === Preprocessing ===
# def preprocess_image(image_path, target_size=(96, 103)):
#     img = load_img(image_path, target_size=target_size, color_mode="grayscale")
#     img_array = img_to_array(img) / 255.0
#     img_array = np.repeat(img_array, 3, axis=-1)  # Make it RGB for EfficientNet
#     return img_array

# def load_data_from_directory(directory, target_size=(96, 103)):
#     X, y = [], []
#     class_labels = sorted(os.listdir(directory))
#     label_map = {label: idx for idx, label in enumerate(class_labels)}
#     print(f"Class mapping: {label_map}")

#     for label in class_labels:
#         label_path = os.path.join(directory, label)
#         for img_file in os.listdir(label_path):
#             img_path = os.path.join(label_path, img_file)
#             try:
#                 X.append(preprocess_image(img_path, target_size))
#                 y.append(label_map[label])
#             except Exception as e:
#                 print(f"Error loading {img_path}: {e}")

#     return np.array(X), np.array(y), label_map

# # === Load Data ===
# X_train, y_train, train_label_map = load_data_from_directory(train_path)
# X_test, y_test, test_label_map = load_data_from_directory(test_path)
# num_classes = len(set(y_train))

# # === Compute Class Weights ===
# class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
# class_weights_dict = {i: w for i, w in enumerate(class_weights)}
# print("Class Weights:", class_weights_dict)

# # === Model Architecture ===
# input_shape = (96, 103, 3)

# def lr_schedule(epoch):
#     initial_lr = 1e-3
#     drop = 0.5
#     epochs_drop = 5
#     return initial_lr * (drop ** (epoch // epochs_drop))

# def build_model(num_classes):
#     base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=input_shape)
#     base_model.trainable = True

#     inputs = Input(shape=input_shape)
#     x = base_model(inputs, training=False)
#     x = GlobalAveragePooling2D()(x)
#     x = BatchNormalization()(x)  # ✅ Added
#     x = Dropout(0.4)(x)
#     outputs = Dense(num_classes, activation='softmax')(x)

#     model = Model(inputs, outputs)
#     model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#                   loss='sparse_categorical_crossentropy',
#                   metrics=['accuracy'])
#     model.summary()
#     return model

# # === Train Model ===
# def build_and_train_model(X_train, y_train, X_test, y_test, num_classes):
#     model = build_model(num_classes)

#     callbacks = [
#         LearningRateScheduler(lr_schedule),
#         EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
#         ModelCheckpoint(model_save_path, monitor="val_accuracy", save_best_only=True, verbose=1)
#     ]

#     history = model.fit(
#         X_train, y_train,
#         validation_data=(X_test, y_test),
#         epochs=50,  # ✅ 50 Epochs
#         batch_size=32,
#         callbacks=callbacks,
#         class_weight=class_weights_dict  # ✅ Class weights
#     )

#     model.evaluate(X_test, y_test, verbose=1)
#     return model, history

# model, history = build_and_train_model(X_train, y_train, X_test, y_test, num_classes)

# # === Plotting ===
# def plot_training_curves(history):
#     plt.figure(figsize=(12, 4))

#     plt.subplot(1, 2, 1)
#     plt.plot(history.history['accuracy'], label='Train Accuracy')
#     plt.plot(history.history['val_accuracy'], label='Val Accuracy')
#     plt.title("Accuracy")
#     plt.xlabel("Epochs")
#     plt.legend()

#     plt.subplot(1, 2, 2)
#     plt.plot(history.history['loss'], label='Train Loss')
#     plt.plot(history.history['val_loss'], label='Val Loss')
#     plt.title("Loss")
#     plt.xlabel("Epochs")
#     plt.legend()

#     plt.tight_layout()
#     plt.show()

# plot_training_curves(history)

# # === Evaluation ===
# y_pred = model.predict(X_test)
# y_pred_classes = np.argmax(y_pred, axis=1)

# print("\nClassification Report:")
# print(classification_report(y_test, y_pred_classes))

# conf_matrix = confusion_matrix(y_test, y_pred_classes)
# ConfusionMatrixDisplay(conf_matrix, display_labels=test_label_map.keys()).plot(cmap="Blues")
# plt.show()

Splitting dataset into train/test...
✅ Dataset split complete.
Class mapping: {'A+': 0, 'A-': 1, 'AB+': 2, 'AB-': 3, 'B+': 4, 'B-': 5, 'O+': 6, 'O-': 7, 'dataset_blood_group': 8}
Class mapping: {'A+': 0, 'A-': 1, 'AB+': 2, 'AB-': 3, 'B+': 4, 'B-': 5, 'O+': 6, 'O-': 7, 'dataset_blood_group': 8}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/30
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 808ms/step - accuracy: 0.5533 - loss: 1.4135
Epoch 1: val_accuracy improved from None to 0.24502, saving model to C:/Users/ganga/saved_models\new_model.keras
150/150 ━━━━━━━━━━━━━━━━━━━━ 196s 908ms/step - accuracy: 0.6737 - loss: 1.0285 - val_accuracy: 0.2450 - val_loss: 2.0299 - learning_rate: 0.0010
Epoch 2/30
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 787ms/step - accuracy: 0.7855 - loss: 0.6384
Epoch 2: val_accuracy did not improve from 0.24502
150/150 ━━━━━━━━━━━━━━━━━━━━ 125s 833ms/step - accuracy: 0.7921 - loss: 0.6146 - val_accuracy: 0.1238 - val_loss: 6.6414 - learning_rate: 0.0010
Epoch 3/30
150/150 ━━━━━